# Restaurant multi-agent system - 3 example scenarios

Four LLM-driven agents (Manager, Order, Kitchen, Billing) share one memory and pass work to each other as messages.
For each scenario we print the **full trace**: which agent acted, which tool it called, and who handed off to whom.

> The outputs saved in this notebook were produced with the **offline MOCK brain** (a rule-based stand-in, *not* the real LLM).
> If `GROQ_API_KEY` is in your `.env`, re-running the cells uses the real Groq model automatically and the traces will differ.

In [1]:
import os, sys
if not os.path.isdir("restaurant"):
    raise RuntimeError("Start Jupyter from the project folder (the one that contains the 'restaurant' folder).")
sys.path.insert(0, os.getcwd())

from restaurant import RestaurantSystem

In [2]:
def run_scenario(title, messages):
    """Start a fresh restaurant session and send each customer message in order."""
    print("=" * 100)
    print(title)
    print("=" * 100)
    system = RestaurantSystem()                      # real Groq brain if a key exists, else the mock
    print("Brain:", system.llm.label, "\n")
    for text in messages:
        result = system.chat(text)
        print(f"----- Turn {result.turn} " + "-" * 80)
        print(result.trace_text)                     # the full multi-agent trace for this turn
        print(f"\nFINAL REPLY TO CUSTOMER: {result.reply}\n")
    return system

## Scenario 1 - taking an order with a special note, then adding to it
Shows: Manager -> Kitchen (stock check) -> Order (add items + note) -> Kitchen (time estimate), and **memory persisting across turns** (the second message only says "table 4").

In [3]:
s1 = run_scenario("SCENARIO 1: order with a note, then add a drink", [
    "Table 4 wants 2 paneer tikka and 1 butter naan, one without onion",
    "Add a cold drink to table 4",
])

SCENARIO 1: order with a note, then add a drink
************************************************************************
  OFFLINE MOCK BRAIN IN USE - this is NOT the real LLM.
  It is a rule-based stand-in so you can test without an API key.
  Put GROQ_API_KEY in .env to use the real model.
************************************************************************
Brain: MOCK BRAIN (offline, rule-based) - NOT the real LLM 

----- Turn 1 --------------------------------------------------------------------------------
[CUSTOMER] customer -> manager: Table 4 wants 2 paneer tikka and 1 butter naan, one without onion
[TOOL CALL] manager (step 1): hand_off(agent='kitchen', request='Check whether we can cook these for table 4: 2 Paneer Tikka, 1 Butter Naan (without onion)')
[HANDOFF] manager -> kitchen: Check whether we can cook these for table 4: 2 Paneer Tikka, 1 Butter Naan (without onion)
    [TOOL CALL] kitchen (step 1): check_stock(item='Paneer Tikka')
    [TOOL RESULT] kitchen: {"ok": t

## Scenario 2 - an out-of-stock dish
Shows: the Kitchen agent flags the dish and suggests an alternative; the Manager only sends the available dishes to the Order agent.

In [4]:
s2 = run_scenario("SCENARIO 2: out-of-stock dish and alternative", [
    "Is the dal makhani available?",
    "Table 2 wants 1 dal makhani and 2 butter naan",
])

SCENARIO 2: out-of-stock dish and alternative
************************************************************************
  OFFLINE MOCK BRAIN IN USE - this is NOT the real LLM.
  It is a rule-based stand-in so you can test without an API key.
  Put GROQ_API_KEY in .env to use the real model.
************************************************************************
Brain: MOCK BRAIN (offline, rule-based) - NOT the real LLM 

----- Turn 1 --------------------------------------------------------------------------------
[CUSTOMER] customer -> manager: Is the dal makhani available?
[TOOL CALL] manager (step 1): hand_off(agent='kitchen', request='Check stock for: Is the dal makhani available?')
[HANDOFF] manager -> kitchen: Check stock for: Is the dal makhani available?
    [TOOL CALL] kitchen (step 1): check_stock(item='Dal Makhani')
    [TOOL RESULT] kitchen: {"ok": true, "item": "Dal Makhani", "in_stock": false, "qty_left": 0, "alternative": {"item": "Dal Tadka", "qty_left": 15}}
    [ANSWER

## Scenario 3 - billing, with and without a discount
Table 4 is a small order (below the Rs.1000 threshold -> no discount). Table 5 is a big order (above it -> 10% discount). The **tool** enforces that rule, not the LLM.

In [5]:
s3 = run_scenario("SCENARIO 3: bills with and without discount", [
    "Table 4 wants 2 paneer tikka and 1 butter naan",
    "Table 5 wants 2 butter chicken, 2 garlic naan, 1 veg biryani and 2 gulab jamun",
    "Give me the bill for table 4",
    "Give me the bill for table 5",
])

SCENARIO 3: bills with and without discount
************************************************************************
  OFFLINE MOCK BRAIN IN USE - this is NOT the real LLM.
  It is a rule-based stand-in so you can test without an API key.
  Put GROQ_API_KEY in .env to use the real model.
************************************************************************
Brain: MOCK BRAIN (offline, rule-based) - NOT the real LLM 

----- Turn 1 --------------------------------------------------------------------------------
[CUSTOMER] customer -> manager: Table 4 wants 2 paneer tikka and 1 butter naan
[TOOL CALL] manager (step 1): hand_off(agent='kitchen', request='Check whether we can cook these for table 4: 2 Paneer Tikka, 1 Butter Naan')
[HANDOFF] manager -> kitchen: Check whether we can cook these for table 4: 2 Paneer Tikka, 1 Butter Naan
    [TOOL CALL] kitchen (step 1): check_stock(item='Paneer Tikka')
    [TOOL RESULT] kitchen: {"ok": true, "item": "Paneer Tikka", "in_stock": true, "qty_lef

## What the shared memory and each agent's private history look like afterwards

In [6]:
import json
snap = s3.memory.snapshot()
print("ORDERS:", json.dumps(snap["orders"], indent=1)[:900], "...\n")
print("BILL STATUS:", {t: b["status"] for t, b in snap["bills"].items()})
print("\nMESSAGE LOG (who told whom what):")
for m in snap["message_log"][:12]:
    print(f"  #{m['id']:>2} turn {m['turn']}  {m['sender']:>8} -> {m['receiver']:<8} [{m['kind']}] {m['content'][:80]}")
print("  ...", len(snap["message_log"]), "messages in total")
print("\nEACH AGENT'S OWN HISTORY + TOOLS:")
for name, agent in s3.agents.items():
    print(f"  {name:<8} history={len(agent.history):>2} messages   tools={list(agent.tools)}")

ORDERS: {
 "4": [
  {
   "item": "Paneer Tikka",
   "qty": 2,
   "note": "",
   "unit_price": 220
  },
  {
   "item": "Butter Naan",
   "qty": 1,
   "note": "",
   "unit_price": 50
  }
 ],
 "5": [
  {
   "item": "Butter Chicken",
   "qty": 2,
   "note": "",
   "unit_price": 320
  },
  {
   "item": "Garlic Naan",
   "qty": 2,
   "note": "",
   "unit_price": 60
  },
  {
   "item": "Veg Biryani",
   "qty": 1,
   "note": "",
   "unit_price": 240
  },
  {
   "item": "Gulab Jamun",
   "qty": 2,
   "note": "",
   "unit_price": 90
  }
 ]
} ...

BILL STATUS: {4: 'prepared', 5: 'prepared'}

MESSAGE LOG (who told whom what):
  # 1 turn 1  customer -> manager  [customer] Table 4 wants 2 paneer tikka and 1 butter naan
  # 2 turn 1   manager -> kitchen  [request] Check whether we can cook these for table 4: 2 Paneer Tikka, 1 Butter Naan
  # 3 turn 1   kitchen -> manager  [response] Paneer Tikka is available. Butter Naan is available.
  # 4 turn 1   manager -> order    [request] Table 4: add 2 Paneer